# Cryptocurrency Security Analysis

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import hashlib
import numpy as np
from collections import defaultdict, deque
import time
import json
import re
from dataclasses import dataclass
from typing import List, Dict, Optional
import struct

@dataclass
class Transaction:
    tx_hash: str
    inputs: List[Dict]
    outputs: List[Dict]
    timestamp: float
    value: float
    gas_price: Optional[float] = None
    contract_address: Optional[str] = None

class CryptoAddressAnalyzer:
    def __init__(self):
        self.address_clusters = defaultdict(set)
        self.transaction_graph = defaultdict(list)
        self.risk_scores = defaultdict(float)
        self.known_malicious = set()
        self.mixing_patterns = {}

    def analyze_address_reuse(self, transactions: List[Transaction]) -> Dict[str, float]:
        address_frequency = defaultdict(int)
        for tx in transactions:
            for inp in tx.inputs:
                address_frequency[inp.get('address', '')] += 1
            for out in tx.outputs:
                address_frequency[out.get('address', '')] += 1

        reuse_scores = {}
        for addr, freq in address_frequency.items():
            if freq > 1:
                reuse_scores[addr] = min(freq / 100.0, 1.0)
        return reuse_scores

    def detect_clustering_heuristics(self, transactions: List[Transaction]) -> Dict[str, List[str]]:
        clusters = defaultdict(set)

        for tx in transactions:
            input_addresses = [inp.get('address', '') for inp in tx.inputs if inp.get('address')]
            if len(input_addresses) > 1:
                base_addr = input_addresses[0]
                for addr in input_addresses[1:]:
                    clusters[base_addr].add(addr)
                    clusters[addr].add(base_addr)

        return {k: list(v) for k, v in clusters.items() if v}

    def calculate_mixing_entropy(self, transaction: Transaction) -> float:
        input_values = [inp.get('value', 0) for inp in transaction.inputs]
        output_values = [out.get('value', 0) for out in transaction.outputs]

        total_in = sum(input_values)
        total_out = sum(output_values)

        if total_out == 0:
            return 0.0

        entropy = 0.0
        for value in output_values:
            if value > 0:
                p = value / total_out
                entropy -= p * np.log2(p)

        return entropy

class SmartContractAnalyzer:
    def __init__(self):
        self.vulnerability_patterns = {
            'reentrancy': [
                r'call\.value\(',
                r'\.call\(',
                r'external.*before.*state'
            ],
            'integer_overflow': [
                r'SafeMath',
                r'\+\+',
                r'unchecked',
                r'uint.*\+'
            ],
            'access_control': [
                r'onlyOwner',
                r'require\(msg\.sender',
                r'modifier.*only',
                r'admin.*check'
            ],
            'timestamp_dependence': [
                r'block\.timestamp',
                r'now\s*[><=]',
                r'block\.number'
            ]
        }

    def analyze_bytecode_patterns(self, bytecode: str) -> Dict[str, float]:
        if not bytecode or not bytecode.startswith('0x'):
            return {}

        risks = {}

        dangerous_opcodes = {
            'CALL': 0.8,
            'DELEGATECALL': 0.9,
            'CALLCODE': 0.9,
            'SELFDESTRUCT': 0.7,
            'CREATE': 0.6,
            'CREATE2': 0.6
        }

        for opcode, risk in dangerous_opcodes.items():
            opcode_hex = hex(hash(opcode) % 256)[2:].zfill(2)
            if opcode_hex in bytecode.lower():
                risks[f'opcode_{opcode.lower()}'] = risk

        complexity_score = len(bytecode) / 10000.0
        risks['complexity'] = min(complexity_score, 1.0)

        return risks

    def detect_ponzi_patterns(self, transactions: List[Transaction]) -> Dict[str, float]:
        if len(transactions) < 10:
            return {}

        early_investors = set()
        total_inflow = 0
        total_outflow = 0

        for i, tx in enumerate(transactions[:5]):
            for inp in tx.inputs:
                early_investors.add(inp.get('address', ''))
            total_inflow += tx.value

        ponzi_score = 0.0

        for tx in transactions[5:]:
            for out in tx.outputs:
                if out.get('address') in early_investors:
                    ponzi_score += out.get('value', 0)
            total_outflow += tx.value

        if total_inflow > 0:
            ponzi_ratio = ponzi_score / total_inflow
            return {'ponzi_suspicion': min(ponzi_ratio * 2, 1.0)}

        return {}

class CryptoForensics:
    def __init__(self):
        self.address_analyzer = CryptoAddressAnalyzer()
        self.contract_analyzer = SmartContractAnalyzer()
        self.transaction_timeline = deque(maxlen=10000)

    def generate_transaction_hash(self, tx_data: dict) -> str:
        tx_string = json.dumps(tx_data, sort_keys=True)
        return hashlib.sha256(tx_string.encode()).hexdigest()

    def detect_wash_trading(self, transactions: List[Transaction]) -> Dict[str, float]:
        address_pairs = defaultdict(list)

        for tx in transactions:
            for inp in tx.inputs:
                for out in tx.outputs:
                    inp_addr = inp.get('address', '')
                    out_addr = out.get('address', '')
                    if inp_addr and out_addr and inp_addr != out_addr:
                        pair = tuple(sorted([inp_addr, out_addr]))
                        address_pairs[pair].append(tx.timestamp)

        wash_scores = {}
        for pair, timestamps in address_pairs.items():
            if len(timestamps) > 5:
                time_diffs = np.diff(sorted(timestamps))
                regularity = np.std(time_diffs) if len(time_diffs) > 1 else float('inf')
                if regularity < 3600:
                    wash_scores[f"{pair[0][:8]}->{pair[1][:8]}"] = min(len(timestamps) / 20.0, 1.0)

        return wash_scores

    def analyze_transaction_velocity(self, address: str, transactions: List[Transaction]) -> Dict[str, float]:
        addr_txs = []
        for tx in transactions:
            involved = False
            for inp in tx.inputs:
                if inp.get('address') == address:
                    involved = True
                    break
            if not involved:
                for out in tx.outputs:
                    if out.get('address') == address:
                        involved = True
                        break
            if involved:
                addr_txs.append(tx.timestamp)

        if len(addr_txs) < 2:
            return {}

        addr_txs.sort()
        intervals = np.diff(addr_txs)
        avg_interval = np.mean(intervals)
        velocity_score = min(3600 / max(avg_interval, 60), 1.0)

        return {
            'velocity_score': velocity_score,
            'avg_interval_hours': avg_interval / 3600,
            'total_transactions': len(addr_txs)
        }

    def detect_mixer_usage(self, transactions: List[Transaction]) -> List[Dict]:
        mixer_indicators = []
        known_mixers = {'tornado', 'mixer', 'tumbler', 'privacy', 'anon'}

        for tx in transactions:
            for out in tx.outputs:
                addr = out.get('address', '').lower()
                for mixer_term in known_mixers:
                    if mixer_term in addr:
                        mixing_entropy = self.address_analyzer.calculate_mixing_entropy(tx)
                        mixer_indicators.append({
                            'tx_hash': tx.tx_hash,
                            'mixer_type': mixer_term,
                            'entropy': mixing_entropy,
                            'value': tx.value,
                            'timestamp': tx.timestamp
                        })
                        break

        return mixer_indicators

    def generate_risk_report(self, address: str, transactions: List[Transaction]) -> Dict:
        report = {
            'address': address,
            'analysis_timestamp': time.time(),
            'total_transactions': len(transactions),
            'risk_factors': {}
        }

        reuse_analysis = self.address_analyzer.analyze_address_reuse(transactions)
        if address in reuse_analysis:
            report['risk_factors']['address_reuse'] = reuse_analysis[address]

        wash_trading = self.detect_wash_trading(transactions)
        if wash_trading:
            report['risk_factors']['wash_trading'] = max(wash_trading.values())

        velocity_analysis = self.analyze_transaction_velocity(address, transactions)
        if velocity_analysis:
            report['risk_factors'].update(velocity_analysis)

        mixer_usage = self.detect_mixer_usage(transactions)
        if mixer_usage:
            report['risk_factors']['mixer_usage'] = len(mixer_usage)
            report['mixer_details'] = mixer_usage

        clustering = self.address_analyzer.detect_clustering_heuristics(transactions)
        if address in clustering:
            report['risk_factors']['clustering_size'] = len(clustering[address])
            report['associated_addresses'] = clustering[address]

        overall_risk = np.mean(list(report['risk_factors'].values())) if report['risk_factors'] else 0.0
        report['overall_risk_score'] = min(overall_risk, 1.0)

        return report

def simulate_crypto_analysis():
    forensics = CryptoForensics()

    sample_transactions = []
    addresses = [f"0x{''.join([f'{i:02x}' for i in range(20)])}" for _ in range(10)]

    for i in range(100):
        tx = Transaction(
            tx_hash=f"0x{hashlib.sha256(f'tx_{i}'.encode()).hexdigest()}",
            inputs=[{'address': np.random.choice(addresses), 'value': np.random.uniform(0.1, 10)}],
            outputs=[{'address': np.random.choice(addresses), 'value': np.random.uniform(0.1, 10)}],
            timestamp=time.time() - np.random.uniform(0, 86400 * 30),
            value=np.random.uniform(0.1, 100),
            gas_price=np.random.uniform(20, 200)
        )
        sample_transactions.append(tx)

    suspicious_addr = addresses[0]

    for i in range(20):
        tx = Transaction(
            tx_hash=f"0x{hashlib.sha256(f'suspicious_{i}'.encode()).hexdigest()}",
            inputs=[{'address': suspicious_addr, 'value': 5.0}],
            outputs=[{'address': addresses[1], 'value': 5.0}],
            timestamp=time.time() - i * 300,
            value=5.0,
            gas_price=50.0
        )
        sample_transactions.append(tx)

    print("=== Cryptocurrency Security Analysis ===\n")

    risk_report = forensics.generate_risk_report(suspicious_addr, sample_transactions)

    print(f"Risk Analysis for Address: {risk_report['address'][:12]}...")
    print(f"Total Transactions Analyzed: {risk_report['total_transactions']}")
    print(f"Overall Risk Score: {risk_report['overall_risk_score']:.3f}")

    print("\n--- Risk Factors ---")
    for factor, score in risk_report['risk_factors'].items():
        severity = "HIGH" if score > 0.7 else "MEDIUM" if score > 0.4 else "LOW"
        print(f"{factor.replace('_', ' ').title()}: {score:.3f} [{severity}]")

    wash_trading = forensics.detect_wash_trading(sample_transactions)
    if wash_trading:
        print(f"\n--- Wash Trading Detection ---")
        for pair, score in list(wash_trading.items())[:3]:
            print(f"Suspicious Pair {pair}: {score:.3f}")

    mixer_usage = forensics.detect_mixer_usage(sample_transactions)
    if mixer_usage:
        print(f"\n--- Privacy Coin/Mixer Usage ---")
        print(f"Detected {len(mixer_usage)} potential mixer transactions")

    contract_risks = forensics.contract_analyzer.analyze_bytecode_patterns("0x608060405234801561001057600080fd5b50")
    if contract_risks:
        print(f"\n--- Smart Contract Analysis ---")
        for risk_type, score in contract_risks.items():
            print(f"{risk_type}: {score:.3f}")

    print(f"\n--- Analysis Complete ---")
    print(f"Framework processed {len(sample_transactions)} transactions")
    print(f"Identified {len([k for k,v in risk_report['risk_factors'].items() if v > 0.5])} high-risk factors")

if __name__ == "__main__":
    simulate_crypto_analysis()